_Authored by Kabelo Tsiane, 2026-10-19_

# Jackknife covariance with HEALPix and k-means regions

`ClusterEnsemble.compute_jackknife_covariance` estimates the covariance of a stacked profile by dividing the sky into regions and removing each region that contains clusters in turn. The regions can be

- **HEALPix pixels** (default), set by `n_side`: a fixed grid, the same for every ensemble, but the pixels do not follow the footprint of the clusters;
- **k-means regions**, set by `n_regions`: regions of similar numbers of clusters that follow the footprint, computed from the cluster positions with `clmm.utils.compute_kmeans_centers`;
- **predefined k-means regions**, set by `centers`: the same regions for several ensembles, e.g. for different redshift or richness bins.

Computing k-means regions requires the optional [kmeans_radec](https://github.com/esheldon/kmeans_radec) package:
```
pip install git+https://github.com/esheldon/kmeans_radec.git
```
Using predefined `centers` does not.

In [ ]:
import warnings

import healpy
import matplotlib.pyplot as plt
import numpy as np

import clmm
from clmm import ClusterEnsemble, Cosmology, GalaxyCluster
from clmm.support import mock_data as mock

clmm.__version__

In [ ]:
np.random.seed(11)
warnings.simplefilter("ignore")  # silence warnings from the mock catalogs

## Mock cluster ensemble

We draw `n_clusters` clusters uniformly in log-mass and redshift, spread over a 40 x 40 deg patch of sky, and generate a background galaxy catalog for each of them.

In [ ]:
cosmo = Cosmology(H0=71.0, Omega_dm0=0.265 - 0.0448, Omega_b0=0.0448, Omega_k0=0.0)

n_clusters = 200
cluster_m = 10 ** np.random.uniform(14.0, 14.5, n_clusters)
cluster_z = np.random.uniform(0.2, 0.4, n_clusters)
concentration = np.full(n_clusters, 4.0)

# uniform on the sphere in 0 < ra < 40 deg, -20 < dec < 20 deg
cluster_ra = np.random.uniform(0.0, 40.0, n_clusters)
sin_dec = np.sin(np.radians(20.0))
cluster_dec = np.degrees(np.arcsin(np.random.uniform(-sin_dec, sin_dec, n_clusters)))

In [ ]:
gclist = []
for m, z, c, ra, dec in zip(cluster_m, cluster_z, concentration, cluster_ra, cluster_dec):
    galcat = mock.generate_galaxy_catalog(
        m,
        z,
        c,
        cosmo,
        "chang13",
        delta_so=200,
        massdef="critical",
        halo_profile_model="nfw",
        zsrc_min=z + 0.1,
        zsrc_max=3.0,
        field_size=10.0,
        shapenoise=0.25,
        ngal_density=5,
        cluster_ra=ra,
        cluster_dec=dec,
    )
    cl = GalaxyCluster(f"mock_{len(gclist)}", ra, dec, z, galcat)
    cl.compute_tangential_and_cross_components(add=True)
    cl.compute_galaxy_weights(cosmo=cosmo, add=True)
    gclist.append(cl)

Individual and stacked tangential shear profiles:

In [ ]:
bins = clmm.dataops.make_bins(0.3, 5.0, nbins=6, method="evenlog10width")
profile_args = dict(bins=bins, bin_units="Mpc", cosmo=cosmo)

ensemble = ClusterEnsemble("all", gclist, **profile_args)
ensemble.make_stacked_radial_profile()
ensemble.data[:3]

## HEALPix regions

With `n_side=8` the pixels are about 7 deg across. Only the pixels that contain clusters are used.

In [ ]:
n_side = 8
ensemble.compute_jackknife_covariance(n_side=n_side)
cov_healpix = ensemble.cov["tan_jk"]

pixels = healpy.ang2pix(n_side, ensemble["ra"], ensemble["dec"], nest=True, lonlat=True)
n_healpix = np.unique(pixels).size
print(f"{n_healpix} occupied HEALPix pixels")

## k-means regions

For a fair comparison we use as many k-means regions as occupied HEALPix pixels. `seed` makes the regions reproducible.

In [ ]:
ensemble.compute_jackknife_covariance(n_regions=n_healpix, seed=11)
cov_kmeans = ensemble.cov["tan_jk"]

# the same regions, to plot them
centers = clmm.utils.compute_kmeans_centers(ensemble["ra"], ensemble["dec"], n_healpix, seed=11)
labels = clmm.utils.assign_kmeans_regions(ensemble["ra"], ensemble["dec"], centers)

Number of clusters in each region. The HEALPix grid is fixed, so its pixels can cut through groups of clusters or be only partly filled at the edges of the footprint, while the k-means regions are built from the cluster positions. Here the clusters are spread uniformly, so both are similar; the difference grows for an irregular footprint:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, regions, title in zip(axes[:2], (pixels, labels), ("HEALPix pixels", "k-means regions")):
    _, regions = np.unique(regions, return_inverse=True)
    ax.scatter(ensemble["ra"], ensemble["dec"], c=regions, cmap="tab20", s=15)
    ax.set_xlabel("ra [deg]")
    ax.set_ylabel("dec [deg]")
    ax.set_title(title)
axes[1].scatter(*centers.T, marker="x", c="k", label="centers")
axes[1].legend()

axes[2].hist(
    [np.unique(pixels, return_counts=True)[1], np.bincount(labels)],
    bins=np.arange(0, 30, 2),
    label=["HEALPix", "k-means"],
)
axes[2].set_xlabel("clusters per region")
axes[2].legend()
fig.tight_layout()

Errors on the stacked tangential shear from both sets of regions:

In [ ]:
radius = ensemble.stacked_data["radius"]
fig, ax = plt.subplots()
ax.plot(radius, np.sqrt(np.diag(cov_healpix)), "o-", label="HEALPix")
ax.plot(radius, np.sqrt(np.diag(cov_kmeans)), "s-", label="k-means")
ax.set_xscale("log")
ax.set_xlabel("R [Mpc]")
ax.set_ylabel(r"$\sigma(g_t)$")
ax.legend()

## Shared regions for several ensembles

When an ensemble is split into bins (here in redshift), the k-means regions can be computed once from all the clusters and passed to every bin with `centers`. Every bin then uses the same regions, as with HEALPix pixels, and the jackknife samples of different bins remove the same part of the sky. Regions with no clusters of a bin are skipped.

In [ ]:
for z_min, z_max in [(0.2, 0.3), (0.3, 0.4)]:
    in_bin = [gc for gc in gclist if z_min <= gc.z < z_max]
    ensemble_bin = ClusterEnsemble(f"z{z_min}-{z_max}", in_bin, **profile_args)
    ensemble_bin.make_stacked_radial_profile()
    ensemble_bin.compute_jackknife_covariance(centers=centers)
    n_occupied = np.unique(
        clmm.utils.assign_kmeans_regions(ensemble_bin["ra"], ensemble_bin["dec"], centers)
    ).size
    print(f"z {z_min}-{z_max}: {len(in_bin)} clusters in {n_occupied} of {len(centers)} regions")